# 01 · Recover a known planet: Pi Mensae c

Goal: show that the pipeline finds a planet we **already know is there** before we trust it on anything else.

Pi Mensae c was the first planet announced from TESS (Huang et al. 2018):

| | Published |
|---|---|
| Period | 6.268 days |
| Depth | ~300 ppm (0.03%) |
| Duration | ~3 hours |

The depth is tiny, only about 3 parts in 10,000, so this is a fair test. We go through the pipeline one step at a time. Every step is a function in `transit_search.py`, so you can read the code next to the output.

In [ ]:
import sys
sys.path.append("..")          # so we can import transit_search.py from the repo root

import numpy as np
import matplotlib.pyplot as plt
import transit_search as ts

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 3.5)

## Step 1: Download

`lightkurve` queries NASA's MAST archive. We ask for **SPOC 2-minute PDCSAP** flux, which NASA's pipeline has already corrected for most spacecraft systematics. We take one sector (~27 days) to start.

In [ ]:
TARGET = "TIC 261136679"     # Pi Mensae. TIC IDs skip the name-lookup service
KNOWN_PERIOD = 6.2678         # days, from the literature

lc = ts.download_light_curve(TARGET, max_sectors=1)
print(f"{len(lc)} cadences, sectors {lc.meta['SECTORS']}, "
      f"{lc.time.value.min():.1f} to {lc.time.value.max():.1f} BTJD")
lc.plot();

Can you see the transits by eye? Probably not: a 300 ppm dip is buried in the scatter. That is why we need a search algorithm.

## Step 2: Clean

`clean_light_curve` does three things (read the docstring for why each matters):
1. drops NaNs and normalises to 1
2. **flattens**: fits a robust curve to each day of data and divides it out, so only short dips remain (transits and flares barely affect the fit)
3. clips outliers *upward only*, so cosmic rays go but transits stay

In [ ]:
normalized, flattened = ts.clean_light_curve(lc, window_days=1.0)

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
normalized.plot(ax=axes[0], title="normalised")
flattened.plot(ax=axes[1], title="flattened + outlier-clipped");

## Step 3: Box Least Squares

BLS tries thousands of periods. For each one it folds the light curve, slides a box-shaped dip across it, and scores how much better "flat + box" fits than "flat". The period with the best score wins.

In [ ]:
first_pass, periodogram, bls = ts.run_bls(flattened, min_period=0.5)
for key in ["period_days", "depth_ppm", "duration_hours", "depth_snr", "sde"]:
    print(f"{key:>15}: {first_pass[key]:.4f}")

### Why we run a second pass, and why it only *measures*

Even a robust trend fit bends a little toward a shallow transit at the centre of its window, so the dip looks **shallower** than it is. The fix: now that we know *when* the transits happen, we mask them and flatten again, so the trend is fitted only to out-of-transit data. Then we re-measure the depth **at the same period and t0**.

We do **not** search again on the masked curve. The data has been shaped around pass 1's choice, so a new search could "re-discover" it, even if it was pure noise. The injection tests in notebook 02 caught exactly this. So detection (period, SDE, `is_candidate`) comes from pass 1 only.

`ts.search_light_curve()` does both passes for you. Here it is by hand:

In [ ]:
period = first_pass["period_days"]
duration = first_pass["duration_hours"] / 24
t0 = first_pass["t0_btjd"]

mask = normalized.create_transit_mask(period=period, transit_time=t0, duration=2 * duration)
normalized, flattened = ts.clean_light_curve(lc, window_days=1.0, transit_mask=mask)
measured = ts.transit_stats(flattened, period, duration, t0)   # measure only; no new search

result = {**first_pass, **measured}
print(f"depth, blind flattening:  {first_pass['depth_ppm']:.0f} ppm")
print(f"depth, masked flattening: {result['depth_ppm']:.0f} ppm")

## Step 4: Plot everything

In [ ]:
fig = ts.plot_diagnostics(normalized, flattened, periodogram, result, title=TARGET)

## Did we recover it?

In [ ]:
print(f"Published period: {KNOWN_PERIOD:.4f} d")
print(f"BLS period:       {result['period_days']:.4f} d  ->  {ts.period_match(result['period_days'], KNOWN_PERIOD)}")
print(f"Depth:            {result['depth_ppm']:.0f} ppm (published ~300)")
print(f"Duration:         {result['duration_hours']:.2f} h (published ~3)")
print(f"SDE / SNR:        {result['sde']:.1f} / {result['depth_snr']:.1f}")
print(f"Odd/even depth difference: {result['odd_even_sigma']:.1f} sigma (small = consistent with a planet)")

## Experiments (to build intuition)

1. **Flatten window too short.** Re-run Step 2 with `window_days=0.15` (3.6 h, about the transit length). What happens to the depth and SDE?
2. **Symmetric clipping.** In `clean_light_curve`, set `sigma_lower=4`. How many in-transit points disappear?
3. **Less data.** Cut the light curve to the first 10 days (`lc[lc.time.value < lc.time.value.min() + 10]`). With only one or two transits, does BLS still lock on?
4. **Other planets.** Try WASP-18 b, `"TIC 100100827"` (deep, 0.94 d), or LHS 3844 b, `"TIC 410153553"` (needs `min_period=0.3`; 30-minute transits).
5. **All at once.** From the repo root, run `python validate_known_planets.py`.

## No internet? Synthetic test

The same pipeline on a fake light curve with an injected Pi Men c-like transit. This is useful for checking the method, but it is **not real data**.

In [ ]:
from tests.test_synthetic import fake_light_curve

fake = fake_light_curve(period=6.27, depth=300e-6, duration_days=3/24)
fake_result = ts.search_light_curve(fake, name="synthetic (injected)", show=True)